# Model Deployment and Inference on DPO Data Generation — v2

Generates a new 3k DPO training dataset with richer negative types.

**Positive side:** reuses `sft_data_ef.jsonl` — teacher-generated justifications that lead with a direct quote from the passage before connecting to the verdict. Stronger grounding than v1 positives.

**Negative taxonomy (6 types, 3000 total):**

| Type | What it targets | Source | Count |
|------|----------------|--------|-------|
| NEG1 — Wrong label | Label correctness | Reuse `dpo_neg1_3k.jsonl` | 750 |
| NEG2 — Bad reasoning | Overreach / fabrication | Reuse `dpo_neg2_3k.jsonl` | 750 |
| NEG3 — Label hedging | Decisiveness | Constructed | 250 |
| NEG4 — Circular citing | Passage grounding | GPT-4.1-mini | 500 |
| NEG5 — Degenerate output | Repetition / garbage | Constructed | 250 |
| NEG6 — Reasoning-label mismatch | Internal consistency | Constructed | 500 |
| **Total** | | | **3000** |

NEG6 is the most important new type: the model reasons correctly to the right verdict but then outputs the wrong label — exactly the failure seen in the dpo_c3 Ernest Medina example. DPO is well-suited to penalize this because the contrast is clear: same justification, right label (chosen) vs wrong label (rejected).

In [ ]:
#!pip install -U openai tqdm
#!pip install azure-identity

In [ ]:
import os
import json
import time
import random
from collections import Counter, defaultdict
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential

RESOURCE_GROUP = "cis-5270-team-10"
OPENAI_API_KEY = ""

OPENAI_ENDPOINT = f"https://{RESOURCE_GROUP}.openai.azure.com"
SUBSCRIPTION_ID = ""

os.environ["AZURE_SUBSCRIPTION_ID"] = SUBSCRIPTION_ID
os.environ["AZURE_RESOURCE_GROUP"]  = "CIS-5270"
os.environ["AZURE_AOAI_ACCOUNT"]    = RESOURCE_GROUP
os.environ["AZURE_OPENAI_API_KEY"]  = OPENAI_API_KEY
os.environ["AZURE_OPENAI_ENDPOINT"] = OPENAI_ENDPOINT

CREDENTIAL = DefaultAzureCredential()

openai_client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    azure_endpoint=OPENAI_ENDPOINT,
    api_version="2025-04-01-preview",
)

TEACHER_DEPLOYMENT = "gpt-4.1-mini"

random.seed(42)

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def save_jsonl(path, rows):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Connected to Azure OpenAI")

Connected to Azure OpenAI


## Imports and helpers

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import json
import time
import random
import re
from collections import Counter, defaultdict

data_dir = "/content/drive/MyDrive"

dpo_v2_path = f"{data_dir}/dpo_data_v2_3k.jsonl"
dev_path = f"{data_dir}/fever_dev_joined.jsonl"

print("DPO v2 path exists:", os.path.exists(dpo_v2_path))
print("DEV path exists:", os.path.exists(dev_path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DPO v2 path exists: True
DEV path exists: True


In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

In [ ]:
dpo_v2_data = load_jsonl(dpo_v2_path)

print("DPO v2 rows:", len(dpo_v2_data))
print("Example keys:", dpo_v2_data[0].keys())
print(json.dumps(dpo_v2_data[0], indent=2)[:1500])

if "neg_type" in dpo_v2_data[0]:
    print("\nNegative type breakdown:")
    print(Counter(row["neg_type"] for row in dpo_v2_data))

DPO v2 rows: 3000
Example keys: dict_keys(['id', 'neg_type', 'prompt', 'chosen', 'rejected'])
{
  "id": 114568,
  "neg_type": "neg1_wrong_label",
  "prompt": "Passage: Dujarric made his official debut for the 2004 Summer Olympics in Athens , where he placed twenty-first in men 's skeet , with a score of 119 points , tying his position with seven other shooters including former Olympic champion Ennio Falco of Italy , and five-time Olympian Guillermo Alfredo Torres of Cuba .\n\nClaim: Faith Evans met Puff Daddy at a Bad Boy photo shoot.",
  "chosen": "NOT MENTIONED: The passage only discusses Dujarric's participation in the 2004 Summer Olympics and does not mention Faith Evans, Puff Daddy, or a Bad Boy photo shoot.",
  "rejected": "SUPPORTED: The passage states that \"It was first broadcast on September 13, 2005, on The WB,\" directly supporting the claim that The WB first aired Supernatural in 2005."
}

Negative type breakdown:
Counter({'neg1_wrong_label': 750, 'neg2_bad_reasoning': 750

In [ ]:
SYSTEM_PROMPT = """You are a fact-checking assistant. Given a passage and a claim, write a one-sentence justification using this structure:
1. Quote or closely paraphrase the most relevant part of the passage.
2. State how that evidence supports the verdict.

Format your full response as: LABEL: justification sentence
Label must be one of: SUPPORTED, CONTRADICTED, NOT MENTIONED

The justification must lead with the evidence, not the conclusion."""

In [ ]:
def convert_row_to_dpo_format(row):
    return {
        "input": {
            "messages": [
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": row["prompt"]
                }
            ]
        },
        "preferred_output": [
            {
                "role": "assistant",
                "content": row["chosen"]
            }
        ],
        "non_preferred_output": [
            {
                "role": "assistant",
                "content": row["rejected"]
            }
        ]
    }

converted_rows = [convert_row_to_dpo_format(row) for row in dpo_v2_data]

print("Converted rows:", len(converted_rows))
print(json.dumps(converted_rows[0], indent=2)[:1500])

Converted rows: 3000
{
  "input": {
    "messages": [
      {
        "role": "system",
        "content": "You are a fact-checking assistant. Given a passage and a claim, write a one-sentence justification using this structure:\n1. Quote or closely paraphrase the most relevant part of the passage.\n2. State how that evidence supports the verdict.\n\nFormat your full response as: LABEL: justification sentence\nLabel must be one of: SUPPORTED, CONTRADICTED, NOT MENTIONED\n\nThe justification must lead with the evidence, not the conclusion."
      },
      {
        "role": "user",
        "content": "Passage: Dujarric made his official debut for the 2004 Summer Olympics in Athens , where he placed twenty-first in men 's skeet , with a score of 119 points , tying his position with seven other shooters including former Olympic champion Ennio Falco of Italy , and five-time Olympian Guillermo Alfredo Torres of Cuba .\n\nClaim: Faith Evans met Puff Daddy at a Bad Boy photo shoot."
      }
  

In [ ]:
cleaned_rows = []

for row in converted_rows:
    try:
        system_msg = row["input"]["messages"][0]["content"].strip()
        user_msg = row["input"]["messages"][1]["content"].strip()
        chosen = row["preferred_output"][0]["content"].strip()
        rejected = row["non_preferred_output"][0]["content"].strip()
    except Exception:
        continue

    if not system_msg or not user_msg or not chosen or not rejected:
        continue
    if chosen == rejected:
        continue

    cleaned_rows.append(row)

print("Usable DPO-v2 rows:", len(cleaned_rows))
print("Dropped rows:", len(converted_rows) - len(cleaned_rows))

Usable DPO-v2 rows: 3000
Dropped rows: 0


In [ ]:
dpo_v2_train_path = f"{data_dir}/dpo_v2_train_azure.jsonl"

save_jsonl(dpo_v2_train_path, cleaned_rows)
save_jsonl("dpo_v2_train_azure.jsonl", cleaned_rows)

print("Saved Drive path:", dpo_v2_train_path)
print("Saved local path: dpo_v2_train_azure.jsonl")
print("Rows:", len(cleaned_rows))

Saved Drive path: /content/drive/MyDrive/dpo_v2_train_azure.jsonl
Saved local path: dpo_v2_train_azure.jsonl
Rows: 3000


## Run finetuning job

In [ ]:
openai_client = AzureOpenAI(

    api_key=OPENAI_API_KEY,

    azure_endpoint=OPENAI_ENDPOINT,

    api_version="2025-04-01-preview",

)

BASE_MODEL = "gpt-4.1-nano-2025-04-14"

print("Azure client ready")

print("Base model:", BASE_MODEL)

Azure client ready
Base model: gpt-4.1-nano-2025-04-14


In [ ]:
from google.colab import userdata
from openai import AzureOpenAI
import os

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY").strip()

print("Endpoint:", OPENAI_ENDPOINT)
print("Key length:", len(OPENAI_API_KEY))
print("Looks like Azure OpenAI endpoint:", ".openai.azure.com" in OPENAI_ENDPOINT)

openai_client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    azure_endpoint=OPENAI_ENDPOINT,
    api_version="2025-04-01-preview",
)

Endpoint: https://cis-5270-team-10.openai.azure.com
Key length: 84
Looks like Azure OpenAI endpoint: True


In [ ]:
print("Uploading DPO-v2 training file...")

with open("dpo_v2_train_azure.jsonl", "rb") as f:
    train_file_v2 = openai_client.files.create(
        file=f,
        purpose="fine-tune"
    )

train_file_v2_id = train_file_v2.id

print("DPO-v2 train file ID:", train_file_v2_id)

Uploading DPO-v2 training file...
DPO-v2 train file ID: file-23ce9a8a74854b30a5aa70f04caf152c


In [ ]:
'''
print(f"Creating DPO-v2 c3 fine-tuning job for {BASE_MODEL}")

dpo_v2_c3_job = openai_client.fine_tuning.jobs.create(
    training_file=train_file_v2_id,
    model=BASE_MODEL,
    method={
        "type": "dpo",
        "dpo": {
            "hyperparameters": {
                "n_epochs": 3,
                "batch_size": 1,
                "learning_rate_multiplier": 1.0
            }
        }
    },
    extra_body={"trainingType": "GlobalStandard"},
    suffix="dpo_v2_c3"
)

print("Job ID:", dpo_v2_c3_job.id)
print("Initial status:", dpo_v2_c3_job.status)
'''

'\nprint(f"Creating DPO-v2 c3 fine-tuning job for {BASE_MODEL}")\n\ndpo_v2_c3_job = openai_client.fine_tuning.jobs.create(\n    training_file=train_file_v2_id,\n    model=BASE_MODEL,\n    method={\n        "type": "dpo",\n        "dpo": {\n            "hyperparameters": {\n                "n_epochs": 3,\n                "batch_size": 1,\n                "learning_rate_multiplier": 1.0\n            }\n        }\n    },\n    extra_body={"trainingType": "GlobalStandard"},\n    suffix="dpo_v2_c3"\n)\n\nprint("Job ID:", dpo_v2_c3_job.id)\nprint("Initial status:", dpo_v2_c3_job.status)\n'

In [ ]:
dpo_v2_c3_job_id = 'ftjob-faa10683157649adb3dea0fcb10aad6f'
current = openai_client.fine_tuning.jobs.retrieve('ftjob-faa10683157649adb3dea0fcb10aad6f')
current.status

'succeeded'

In [ ]:
job_id = 'ftjob-faa10683157649adb3dea0fcb10aad6f'

events = openai_client.fine_tuning.jobs.list_events(job_id, limit=100)

for e in events.data:

    msg = getattr(e, "message", "")

    if "step" in msg.lower() or "loss" in msg.lower() or "epoch" in msg.lower():

        print(msg)

Step 9000: training loss=5.53131103515625E-05
Step 8999: training loss=3.719329833984375E-05
Step 8998: training loss=1.239776611328125E-05
Step 8997: training loss=1.049041748046875E-05
Step 8996: training loss=4.100799560546875E-05
Step 8995: training loss=1.049041748046875E-05
Step 8994: training loss=9.5367431640625E-06
Step 8993: training loss=9.5367431640625E-06
Step 8992: training loss=9.5367431640625E-06
Step 8991: training loss=0.003311634063720703
Step 8990: training loss=1.049041748046875E-05
Step 8989: training loss=9.5367431640625E-06
Step 8988: training loss=8.58306884765625E-06
Step 8987: training loss=9.5367431640625E-06
Step 8986: training loss=1.049041748046875E-05
Step 8985: training loss=0.6639635562896729
Step 8984: training loss=2.384185791015625E-05
Step 8983: training loss=5.054473876953125E-05
Step 8982: training loss=7.152557373046875E-05
Step 8981: training loss=5.91278076171875E-05
Step 8980: training loss=1.1444091796875E-05
Step 8979: training loss=6.00814

In [ ]:
dpo_v2_c3_job = current
dpo_v2_c3_job.id

'ftjob-faa10683157649adb3dea0fcb10aad6f'

In [ ]:
job_id_path = f"{data_dir}/dpo_v2_c3_job_id.txt"

with open(job_id_path, "w") as f:
    f.write(dpo_v2_c3_job.id)

print("Saved job ID to:", job_id_path)
print("Job ID:", dpo_v2_c3_job.id)

Saved job ID to: /content/drive/MyDrive/dpo_v2_c3_job_id.txt
Job ID: ftjob-faa10683157649adb3dea0fcb10aad6f


In [ ]:
def wait_for_job(job_id, poll_seconds=60):
    while True:
        current = openai_client.fine_tuning.jobs.retrieve(job_id)
        print(f"{job_id}: {current.status}")

        if current.status in {"succeeded", "failed", "cancelled"}:
            return current

        time.sleep(poll_seconds)

final_dpo_v2_c3_job = wait_for_job(dpo_v2_c3_job_id, poll_seconds=60)

ftjob-faa10683157649adb3dea0fcb10aad6f: succeeded


In [ ]:
final_dpo_v2_c3_job = dpo_v2_c3_job
print("Final status:", final_dpo_v2_c3_job.status)
print("Model:", getattr(final_dpo_v2_c3_job, "model", None))
print("Fine-tuned model:", getattr(final_dpo_v2_c3_job, "fine_tuned_model", None))
print("Error:", getattr(final_dpo_v2_c3_job, "error", None))
print("Training file:", getattr(final_dpo_v2_c3_job, "training_file", None))

if final_dpo_v2_c3_job.status == "succeeded":
    WINNING_DPO_V2_C3_MODEL = final_dpo_v2_c3_job.fine_tuned_model
    print("DPO-v2 c3 fine-tuned model:", WINNING_DPO_V2_C3_MODEL)
else:
    WINNING_DPO_V2_C3_MODEL = None

Final status: succeeded
Model: gpt-4.1-nano-2025-04-14
Fine-tuned model: gpt-4.1-nano-2025-04-14.ft-faa10683157649adb3dea0fcb10aad6f-dpo_v2_c3
Error: None
Training file: file-08b29d41a1c34e59a63f81cf66c3270c
DPO-v2 c3 fine-tuned model: gpt-4.1-nano-2025-04-14.ft-faa10683157649adb3dea0fcb10aad6f-dpo_v2_c3


## model deployment for inference

In [ ]:
DPO_V2_C3_DEPLOYMENT = "1-nano-2025-04-14-dpo_v2_c3"

In [ ]:
from collections import Counter, defaultdict
import re

VALID_LABELS = ["SUPPORTED", "CONTRADICTED", "NOT MENTIONED"]

def get_label_from_response(text):
    if not isinstance(text, str):
        return None
    t = text.upper().strip()
    for lbl in VALID_LABELS:
        if t.startswith(lbl):
            return lbl
    return None

chosen_labels = Counter()
rejected_labels = Counter()

for row in dpo_v2_data:
    chosen_labels[get_label_from_response(row["chosen"])] += 1
    rejected_labels[get_label_from_response(row["rejected"])] += 1

print("Chosen label distribution:")
print(chosen_labels)

print("\nRejected label distribution:")
print(rejected_labels)

Chosen label distribution:
Counter({'NOT MENTIONED': 1007, 'CONTRADICTED': 1000, 'SUPPORTED': 993})

Rejected label distribution:
Counter({'CONTRADICTED': 909, 'NOT MENTIONED': 861, 'SUPPORTED': 840, None: 390})


In [ ]:
by_type = defaultdict(lambda: {"chosen": Counter(), "rejected": Counter(), "count": 0})

for row in dpo_v2_data:
    neg_type = row.get("neg_type", "UNKNOWN")
    by_type[neg_type]["count"] += 1
    by_type[neg_type]["chosen"][get_label_from_response(row["chosen"])] += 1
    by_type[neg_type]["rejected"][get_label_from_response(row["rejected"])] += 1

for neg_type, stats in by_type.items():
    print("\n" + "=" * 80)
    print("NEG TYPE:", neg_type)
    print("Count:", stats["count"])
    print("Chosen:", stats["chosen"])
    print("Rejected:", stats["rejected"])


NEG TYPE: neg1_wrong_label
Count: 750
Chosen: Counter({'NOT MENTIONED': 264, 'SUPPORTED': 257, 'CONTRADICTED': 229})
Rejected: Counter({'CONTRADICTED': 274, 'SUPPORTED': 238, 'NOT MENTIONED': 238})

NEG TYPE: neg4_circular
Count: 500
Chosen: Counter({'CONTRADICTED': 186, 'SUPPORTED': 160, 'NOT MENTIONED': 154})
Rejected: Counter({'CONTRADICTED': 186, 'NOT MENTIONED': 154, 'SUPPORTED': 151, None: 9})

NEG TYPE: neg6_mismatch
Count: 500
Chosen: Counter({'CONTRADICTED': 175, 'NOT MENTIONED': 174, 'SUPPORTED': 151})
Rejected: Counter({'NOT MENTIONED': 176, 'CONTRADICTED': 169, 'SUPPORTED': 155})

NEG TYPE: neg2_bad_reasoning
Count: 750
Chosen: Counter({'SUPPORTED': 254, 'NOT MENTIONED': 252, 'CONTRADICTED': 244})
Rejected: Counter({'SUPPORTED': 254, 'NOT MENTIONED': 252, 'CONTRADICTED': 244})

NEG TYPE: neg5_degenerate
Count: 250
Chosen: Counter({'SUPPORTED': 84, 'NOT MENTIONED': 83, 'CONTRADICTED': 83})
Rejected: Counter({None: 131, 'SUPPORTED': 42, 'NOT MENTIONED': 41, 'CONTRADICTED': 3

In [ ]:
pair_counts = Counter()

for row in dpo_v2_data:
    c = get_label_from_response(row["chosen"])
    r = get_label_from_response(row["rejected"])
    pair_counts[(c, r)] += 1

print("Chosen → Rejected label pairs:")
for pair, count in pair_counts.most_common():
    print(pair, count)

Chosen → Rejected label pairs:
('CONTRADICTED', 'CONTRADICTED') 466
('NOT MENTIONED', 'NOT MENTIONED') 447
('SUPPORTED', 'SUPPORTED') 447
('NOT MENTIONED', 'CONTRADICTED') 233
('CONTRADICTED', 'NOT MENTIONED') 216
('SUPPORTED', 'CONTRADICTED') 210
('NOT MENTIONED', 'SUPPORTED') 205
('SUPPORTED', 'NOT MENTIONED') 198
('CONTRADICTED', 'SUPPORTED') 188
('SUPPORTED', None) 138
('CONTRADICTED', None) 130
('NOT MENTIONED', None) 122


In [ ]:
random.seed(42)

dev_data = load_jsonl(dev_path)

buckets = defaultdict(list)
for ex in dev_data:
    buckets[ex["label"]].append(ex)

eval_sample = []
for lbl, items in buckets.items():
    random.shuffle(items)
    eval_sample.extend(items[:400])  # ~1200 total

random.shuffle(eval_sample)

print("Dev sample:", len(eval_sample))
print(Counter(ex["label"] for ex in eval_sample))

Dev sample: 1200
Counter({'CONTRADICTED': 400, 'SUPPORTED': 400, 'NOT MENTIONED': 400})


In [ ]:
VALID_LABELS = {"SUPPORTED", "CONTRADICTED", "NOT MENTIONED"}

def extract_label(text):

    if not isinstance(text, str) or not text.strip():

        return None

    t = text.upper().strip()

    # Remove leading punctuation/markdown

    t = re.sub(r'^[\s\-\*\#\:\"]+', '', t)

    # Normalize repeated / malformed tokens

    t = t.replace("CONCON", "CON")

    t = re.sub(r"\bCON\s+CON\b", "CON", t)

    t = re.sub(r"\bNOT\s+MENTION\b", "NOT MENTIONED", t)

    # Look only at the beginning first

    prefix = " ".join(t.split()[:8])

    # Strong beginning-based rules

    if re.match(r"^(SUPPORTED|SUPPORT|SUPPORTS|SUPPORTED,|SUPPORTED:)\b", prefix):

        return "SUPPORTED"

    if re.match(r"^(CONTRADICTED|CONTRADICTS|CONTRADICT|REFUTED|REFUTES|REFUTE)\b", prefix):

        return "CONTRADICTED"

    # Handle malformed DPO outputs like "CON, ..." or "CON passage..."

    if re.match(r"^(CON\b|CON,|CON:|CON\s)", prefix):

        return "CONTRADICTED"

    if re.match(r"^(NOT MENTIONED|NOT\s+MENTION|NOT\s+M\b|NEI|NOT ENOUGH INFO)\b", prefix):

        return "NOT MENTIONED"

    # JSON-ish outputs near beginning

    json_label_match = re.search(

        r'["\']?LABEL["\']?\s*[:=]\s*["\']?(SUPPORTED|CONTRADICTED|REFUTED|NOT MENTIONED|NOT ENOUGH INFO|NEI)',

        t[:200]

    )

    if json_label_match:

        label = json_label_match.group(1)

        if label == "REFUTED":

            return "CONTRADICTED"

        if label in {"NOT ENOUGH INFO", "NEI"}:

            return "NOT MENTIONED"

        return label

    # Natural language near beginning only

    first_200 = t[:200]

    if re.search(r"\b(CLAIM|STATEMENT)\s+IS\s+SUPPORTED\b", first_200):

        return "SUPPORTED"

    if re.search(r"\b(CLAIM|STATEMENT)\s+IS\s+(CONTRADICTED|REFUTED)\b", first_200):

        return "CONTRADICTED"

    if re.search(r"\b(CLAIM|STATEMENT)\s+IS\s+(NOT MENTIONED|NOT STATED|NOT ENOUGH INFO)\b", first_200):

        return "NOT MENTIONED"

    if re.search(r"\bDOES NOT MENTION\b|\bNO INFORMATION\b|\bNOT STATED\b", first_200):

        return "NOT MENTIONED"

    # Last resort: only use first explicit valid label occurrence,

    # not generic words like "supports" anywhere later.

    label_positions = []

    patterns = {

        "SUPPORTED": r"\bSUPPORTED\b",

        "CONTRADICTED": r"\bCONTRADICTED\b|\bREFUTED\b",

        "NOT MENTIONED": r"\bNOT MENTIONED\b|\bNOT ENOUGH INFO\b|\bNEI\b",

    }

    for label, pattern in patterns.items():

        m = re.search(pattern, t)

        if m:

            label_positions.append((m.start(), label))

    if label_positions:

        label_positions.sort()

        return label_positions[0][1]

    return None

def predict_dpo(passage, claim, deployment, temperature=0.0):
    user_msg = f"Passage: {passage}\n\nClaim: {claim}"
    for attempt in range(3):
        try:
            resp = openai_client.chat.completions.create(
                model=deployment,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg},
                ],
                temperature=temperature,
                max_tokens=150,
            )
            text = resp.choices[0].message.content.strip()
            label = extract_label(text)
            if label:
                return label, text
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)
            else:
                print(f"  failed after 3 attempts: {e}")
    return None, None

def run_eval(cfg_name, deployment, examples, out_path):
    done_ids = set()
    results = []
    if os.path.exists(out_path):
        with open(out_path) as f:
            for line in f:
                r = json.loads(line)
                done_ids.add(r["id"])
                results.append(r)
        print(f"resuming, {len(done_ids)} already done")

    skipped = 0
    with open(out_path, "a") as out_f:
        for i, ex in enumerate(examples):
            if ex["id"] in done_ids:
                continue
            pred_label, raw = predict_dpo(ex["passage"], ex["claim"], deployment)
            if pred_label is None:
                skipped += 1
                continue
            record = {"id": ex["id"], "label": ex["label"], "pred_label": pred_label, "raw": raw}
            out_f.write(json.dumps(record) + "\n")
            out_f.flush()
            results.append(record)
            if (i + 1) % 100 == 0:
                print(f"  [{i+1}/{len(examples)}] skipped={skipped}")

    print(f"done. total={len(results)} skipped={skipped}")
    return results

In [ ]:
DPO_DEPLOYMENTS = {
    "dpo_c3": "1-nano-2025-04-14-dpo_v2_c3"
}

In [ ]:
all_results = {}
for cfg_name, deployment in DPO_DEPLOYMENTS.items():
    print(f"Config: {cfg_name}  deployment: {deployment}")
    out_path = f"{data_dir}/dpo_v2_c3_dev.jsonl"
    results = run_eval(cfg_name, deployment, eval_sample, out_path)
    all_results[cfg_name] = results
    print()

Config: dpo_c3  deployment: 1-nano-2025-04-14-dpo_v2_c3
resuming, 623 already done
  failed after 3 attempts: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': True, 'severity': 'medium'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}
  [700/1200] skipped=1
  failed after 3 attempts: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's con

In [ ]:
def compute_metrics(results):
    labels = ["SUPPORTED", "CONTRADICTED", "NOT MENTIONED"]
    per_class = {lbl: {"correct": 0, "total": 0} for lbl in labels}
    confusion = {true: {pred: 0 for pred in labels} for true in labels}
    total_correct = 0

    for r in results:
        gt, pred = r["label"], r["pred_label"]
        per_class[gt]["total"] += 1
        if pred in labels:
            confusion[gt][pred] += 1
        if pred == gt:
            per_class[gt]["correct"] += 1
            total_correct += 1

    macro_acc = sum(
        per_class[lbl]["correct"] / per_class[lbl]["total"]
        for lbl in labels if per_class[lbl]["total"] > 0
    ) / len(labels)
    overall_acc = total_correct / len(results)

    print(f"  overall accuracy: {overall_acc:.3f}  ({total_correct}/{len(results)})")
    print(f"  macro accuracy:   {macro_acc:.3f}")
    print()
    print("  Per-class accuracy:")
    for lbl in labels:
        c = per_class[lbl]
        acc = c["correct"] / c["total"] if c["total"] else 0
        print(f"    {lbl:20s}: {acc:.3f}  ({c['correct']}/{c['total']})")
    print()
    print("  Confusion matrix (rows=true, cols=pred):")
    col_w = 14
    print(" " * 22 + "".join(f"{lbl[:col_w]:>{col_w}}" for lbl in labels))
    for true_lbl in labels:
        row = f"  {true_lbl:20s}" + "".join(
            f"{confusion[true_lbl][pred_lbl]:>{col_w}}" for pred_lbl in labels
        )
        print(row)

    return macro_acc

# reload from disk so cell is self-contained
all_results = {}
for cfg_name in DPO_DEPLOYMENTS:
    path = f"{data_dir}/dpo_v2_c3_dev.jsonl"
    if os.path.exists(path):
        records = []
        with open(path) as f:
            for line in f:
                if not line.strip(): continue
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
        all_results[cfg_name] = records
        print(f"loaded {cfg_name}: {len(records)} records")
    else:
        print(f"missing: {path}")

print("\nDPO Dev Results\n")
best_cfg, best_acc = None, 0
for cfg_name, results in all_results.items():
    print(f"Config: {cfg_name}")
    acc = compute_metrics(results)
    if acc > best_acc:
        best_acc, best_cfg = acc, cfg_name
    print()

print(f"Best config: {best_cfg}  (macro accuracy={best_acc:.3f})")

loaded dpo_c3: 1197 records

DPO Dev Results

Config: dpo_c3
  overall accuracy: 0.602  (721/1197)
  macro accuracy:   0.602

  Per-class accuracy:
    SUPPORTED           : 0.848  (339/400)
    CONTRADICTED        : 0.957  (382/399)
    NOT MENTIONED       : 0.000  (0/398)

  Confusion matrix (rows=true, cols=pred):
                           SUPPORTED  CONTRADICTED NOT MENTIONED
  SUPPORTED                      339            61             0
  CONTRADICTED                    17           382             0
  NOT MENTIONED                    0           398             0

Best config: dpo_c3  (macro accuracy=0.602)


In [41]:
best_records = {}
with open(f"{data_dir}/dpo_v2_c3_dev.jsonl") as f:
    for line in f:
        if not line.strip(): continue
        r = json.loads(line)
        best_records[r["id"]] = r

dev_by_id = {}
with open(f"{data_dir}/fever_dev_joined.jsonl") as f:
    for line in f:
        ex = json.loads(line)
        dev_by_id[ex["id"]] = ex

LABELS = ["SUPPORTED", "CONTRADICTED", "NOT MENTIONED"]

for true_label in LABELS:
    correct   = [r for r in best_records.values() if r["label"] == true_label and r["pred_label"] == true_label][:2]
    incorrect = [r for r in best_records.values() if r["label"] == true_label and r["pred_label"] != true_label][:2]

    print(f"{'='*70}")
    print(f"TRUE LABEL: {true_label}")
    print(f"{'='*70}")

    print("\n--- Correct predictions ---")
    for r in correct:
        ex = dev_by_id.get(r["id"], {})
        print(f"  Passage : {ex.get('passage','')[:120]}...")
        print(f"  Claim   : {ex.get('claim','')}")
        print(f"  Output  : {r['raw']}")
        print()

    print("--- Incorrect predictions ---")
    for r in incorrect:
        ex = dev_by_id.get(r["id"], {})
        print(f"  Passage : {ex.get('passage','')[:120]}...")
        print(f"  Claim   : {ex.get('claim','')}")
        print(f"  Predicted: {r['pred_label']}  (true: {r['label']})")
        print(f"  Output  : {r['raw']}")
        print()
    print()

TRUE LABEL: SUPPORTED

--- Correct predictions ---
  Passage : Rates of endometrial cancer have risen in a number of countries between the 1980s and 2010 ....
  Claim   : The rate of endometrial cancer has changed.
  Output  : Supported, passage supports the passage that supports the passage supports for the passage supports for the passage supports for the passage supports for the passage supports for the passage. The passage passage supports the passage supports for the passage supports for the passage passage for the passage passage itself supports the passage passage itself supports the passage supports for the passage itself supports the passage passage itself is a passage for the passage itself passage for the passage itself. Con passage passage supports the passage passage for the passage for the passage passage supports the CON CONCON CON CON CON CON supports the passage passage for the passage passage supports for the passage passage for the passage passage supports the passag

In [ ]:
# Deletion cell for failed results
'''
dpo_v2_c3_dev_path = f"{data_dir}/dpo_v2_c3_dev.jsonl"

if os.path.exists(dpo_v2_c3_dev_path):

    os.remove(dpo_v2_c3_dev_path)

    print("Deleted partial results:", dpo_v2_c3_dev_path)

else:

    print("No previous results file found.")
'''

No previous results file found.
